# 04 · Preprocessing — the anti-leakage pipeline

**DSP391m · Group 1 · FPT University** — Tasks 16–22.

Self-contained (no `src/` imports). The cardinal rule is **fit on train, apply to test**: the train/test split happens *first*, then medians, winsorize thresholds and the scaler are learned on `X_train` only and re-applied to `X_test` — so no test information ever leaks into a learned parameter.

```
split (outside) → handle_missing → handle_outliers → fit_transform(train) → transform(test)
                                → [then SMOTE/ADASYN on X_train only]
```

In [ ]:
import os, json, logging, warnings
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('nb')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
CHECKPOINTS_DIR = ROOT / 'data' / 'checkpoints'
CHECKPOINT_MAP_PATH = ROOT / 'data' / 'checkpoint_map.csv'
REPORTS_DIR = ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'
FIGURES_DIR = REPORTS_DIR / 'figures'
CHECKPOINTS = (10, 20, 40, 60, 80, 100)
RANDOM_SEED = 42
for _d in (INTERIM_DIR, CHECKPOINTS_DIR, TABLES_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

## 0. Variable-type catalogue & safe X/y split (Task 16)

`FEATURE_COLUMNS` is an **allow-list**: the only columns a model may learn from. `LEAKY_COLUMNS` (`at_risk`, `final_result`, `date_unregistration`) can never enter X — `make_X_y` enforces this even if someone does `df.drop(columns=['at_risk'])` by hand.

In [ ]:
NUMERIC_FEATURES: list[str] = [
    # Nhân khẩu học
    "num_of_prev_attempts",  # discrete  – số lần thử lại môn, min=0 max=6
    "studied_credits",  # discrete  – tín chỉ đăng ký, min=30 max=655
    "date_registration",  # continuous– ngày đăng ký (tương đối, có thể âm)
    # Tương tác VLE (phái sinh từ studentVle)
    "total_clicks",  # continuous– tổng lượt click tới mốc t
    "n_days_active",  # discrete  – số ngày có hoạt động tới mốc t
    "clicks_forumng",  # continuous– click loại forumng
    "clicks_oucontent",  # continuous– click loại oucontent
    "clicks_resource",  # continuous– click loại resource
    "clicks_homepage",  # continuous– click loại homepage
    "clicks_oucollaborate",  # continuous– click loại oucollaborate
    "clicks_quiz",  # continuous– click loại quiz
    "clicks_subpage",  # continuous– click loại subpage
    "clicks_url",  # continuous– click loại url
    # Tương tác VLE – đặc trưng phái sinh bổ sung (từ handoff Bình → Đức)
    "max_clicks_single_day",  # continuous– click tối đa trong 1 ngày tới mốc t
    "mean_clicks_per_active_day",  # continuous– trung bình click/ngày có hoạt động
    "days_since_last_activity",  # continuous– số ngày từ lần tương tác cuối tới mốc t
    # Kết quả học tập (phái sinh từ studentAssessment)
    "mean_score_to_date",  # continuous– điểm trung bình các bài đã nộp [0–100]
    "n_assessments_submitted",  # discrete  – số bài đã nộp tới mốc t
    "weighted_score_to_date",  # continuous– tổng (score × weight) bài đã nộp
]


ORDINAL_FEATURES: list[str] = ["highest_education", "imd_band", "age_band"]


ORDINAL_ORDERS: dict[str, list[str]] = {
    "highest_education": [
        "No Formal quals",  # 0
        "Lower Than A Level",  # 1
        "A Level or Equivalent",  # 2
        "HE Qualification",  # 3
        "Post Graduate Qualification",  # 4
    ],
    # 'Unknown' xếp hạng 0 (thêm khi imd_band khuyết)
    "imd_band": [
        "Unknown",  # 0  ← giá trị điền vào ô khuyết
        "0-10%",  # 1
        "10-20",  # 2  ← chú ý: OULAD viết thiếu dấu '%'
        "20-30%",  # 3
        "30-40%",  # 4
        "40-50%",  # 5
        "50-60%",  # 6
        "60-70%",  # 7
        "70-80%",  # 8
        "80-90%",  # 9
        "90-100%",  # 10
    ],
    "age_band": [
        "0-35",  # 0
        "35-55",  # 1
        "55<=",  # 2
    ],
}


NOMINAL_FEATURES: list[str] = ["region", "code_module", "code_presentation"]


BINARY_FEATURES: list[str] = ["gender", "disability"]


INDICATOR_FEATURES: list[str] = ["not_submitted"]


TARGET_COL = "at_risk"


ID_COLS: list[str] = ["id_student", "code_module", "code_presentation"]


FEATURE_COLUMNS: list[str] = (
    NUMERIC_FEATURES
    + ORDINAL_FEATURES
    + NOMINAL_FEATURES
    + BINARY_FEATURES
    + INDICATOR_FEATURES
)


LEAKY_COLUMNS: frozenset[str] = frozenset(
    {TARGET_COL, "final_result", "date_unregistration"}
)


def make_X_y(df: pd.DataFrame) -> tuple[pd.DataFrame, "pd.Series | None"]:
    """Tách an toàn (X, y) từ bảng master/checkpoint.

    X chỉ chứa các cột trong FEATURE_COLUMNS (allow-list), nên các cột rò rỉ
    (`final_result`, `date_unregistration`, nhãn `at_risk`) KHÔNG BAO GIỜ lọt vào
    X — kể cả khi ai đó lỡ tay làm ``X = df.drop(columns=["at_risk"])``. ``y`` là
    cột ``at_risk`` khi có mặt.
    """
    leaked = LEAKY_COLUMNS & set(FEATURE_COLUMNS)
    assert not leaked, f"FEATURE_COLUMNS chứa cột rò rỉ: {sorted(leaked)}"
    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        raise KeyError(f"Thiếu cột đặc trưng: {missing}")
    X = df[FEATURE_COLUMNS].copy()
    y = df[TARGET_COL].copy() if TARGET_COL in df.columns else None
    return X, y

## 1. Missing values (Task 17)

`imd_band` → `Unknown`; score/submission gaps → 0 (no work done yet, plus the `not_submitted` flag); `date_registration` → **train** median (passed via `stats` and re-used on test).

In [ ]:
def log_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    In bảng thống kê giá trị khuyết trước khi xử lý (dùng trong notebook).
    Trả về DataFrame thống kê để tiện lưu.
    """
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    report = pd.DataFrame({"count": miss, "pct_%": pct})[miss > 0].sort_values(
        "pct_%", ascending=False
    )
    print("=== Giá trị khuyết trước khi xử lý ===")
    print(report.to_string() if len(report) > 0 else "  Không có giá trị khuyết.")
    return report


def handle_missing(
    df: pd.DataFrame,
    log_path: Optional[str] = None,
    stats: Optional[dict] = None,
) -> pd.DataFrame:
    """
    Xử lý giá trị khuyết theo từng biến:

    imd_band (MAR/MCAR – 1 111 bản ghi)
        → điền 'Unknown'; thêm vào đầu ORDINAL_ORDERS để encoder nhận diện.

    mean_score_to_date, weighted_score_to_date, n_assessments_submitted
        → khuyết do chưa nộp bài tới mốc t  (cơ chế MNAR)
        → điền 0  (không có điểm tích luỹ = 0)
        → CÒN tạo biến chỉ báo 'not_submitted' ở bước feature engineering.

    date_registration (rất ít hoặc không khuyết)
        → điền median của tập huấn luyện.

    Tham số
    -------
    df       : DataFrame (thường là X_train hoặc X_test SAU khi đã phân chia)
    log_path : nếu không None, ghi CSV nhật ký vào đường dẫn này
    stats    : dict tham số học trên train (chống rò rỉ). Lần gọi đầu (train) để
               trống → hàm TÍNH median và LƯU vào dict; lần gọi sau (test) truyền
               lại dict → hàm DÙNG median train, KHÔNG tính median trên test.
               Bỏ trống (None) → hành vi cũ: tính trên chính df.

    Trả về DataFrame đã xử lý.
    """
    df = df.copy()
    log_dict: dict[str, dict] = {}

    # ── 1. imd_band ──────────────────────────────────────────────────
    col = "imd_band"
    if col in df.columns:
        n = int(df[col].isnull().sum())
        log_dict[col] = {"n_khuyết_trước": n, "chiến_lược": "fill 'Unknown'"}
        df[col] = df[col].fillna("Unknown")
        # Bảo đảm 'Unknown' đứng đầu danh sách thứ tự ordinal
        if "Unknown" not in ORDINAL_ORDERS["imd_band"]:
            ORDINAL_ORDERS["imd_band"].insert(0, "Unknown")

    # ── 2. Các biến điểm số & số bài nộp (MNAR → 0) ─────────────────
    for col in (
        "mean_score_to_date",
        "weighted_score_to_date",
        "n_assessments_submitted",
    ):
        if col in df.columns:
            n = int(df[col].isnull().sum())
            log_dict[col] = {
                "n_khuyết_trước": n,
                "chiến_lược": "fill 0 (chưa nộp bài tới mốc t)",
            }
            df[col] = df[col].fillna(0.0)

    # ── 3. date_registration (median HỌC TRÊN TRAIN, áp cho cả test) ─
    col = "date_registration"
    if col in df.columns:
        n = int(df[col].isnull().sum())
        # FIT (train): tính & lưu median train. APPLY (test): dùng lại median train.
        if stats is not None and "date_registration_median" in stats:
            median_val = stats["date_registration_median"]
        else:
            median_val = float(df[col].median())
            if stats is not None:
                stats["date_registration_median"] = median_val
        if n > 0:
            log_dict[col] = {
                "n_khuyết_trước": n,
                "chiến_lược": f"fill median train ({median_val:.1f})",
            }
            df[col] = df[col].fillna(median_val)

    # ── Kiểm tra sau xử lý ───────────────────────────────────────────
    all_feat_cols = [
        c
        for c in (
            NUMERIC_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES + BINARY_FEATURES
        )
        if c in df.columns
    ]
    remaining = df[all_feat_cols].isnull().sum()
    remaining = remaining[remaining > 0]
    if len(remaining):
        log.warning("Còn giá trị khuyết sau handle_missing: %s", remaining.to_dict())
    else:
        log.info("✔ handle_missing: df.isnull().sum() = 0 ở mọi biến đặc trưng")

    # ── Ghi nhật ký ──────────────────────────────────────────────────
    log_df = pd.DataFrame(log_dict).T
    if log_path:
        log_df.to_csv(log_path, encoding="utf-8-sig")
        log.info("Nhật ký missing → %s", log_path)
    else:
        log.info("Nhật ký xử lý khuyết:\n%s", log_df.to_string())

    return df

## 2. Outliers (Task 18)

No rows are deleted. Strongly right-skewed clicks → `log1p`; milder features → winsorize to [p1, p99] **thresholds learned on train**; bounded features → left alone.

In [ ]:
OUTLIER_STRATEGY: dict[str, str] = {
    # VLE clicks – lệch phải rất mạnh
    "total_clicks": "log1p",
    "n_days_active": "log1p",
    "clicks_forumng": "log1p",
    "clicks_oucontent": "log1p",
    "clicks_resource": "log1p",
    "clicks_homepage": "log1p",
    "clicks_oucollaborate": "log1p",
    "clicks_quiz": "log1p",
    "clicks_subpage": "log1p",
    "clicks_url": "log1p",
    # VLE – đặc trưng phái sinh mới (từ handoff Bình)
    "max_clicks_single_day": "log1p",  # max=7920, rất lệch phải
    "mean_clicks_per_active_day": "log1p",  # max=1879, rất lệch phải
    "days_since_last_activity": "winsorize",  # chỉ 6 bản ghi, lệch nhẹ
    # Nhân khẩu học
    "studied_credits": "winsorize",  # max=655, lệch phải vừa
    "num_of_prev_attempts": "winsorize",  # IQR=0 → can_tren=0; winsorize(1%) giữ tín hiệu
    # Kết quả học tập
    "mean_score_to_date": "none",  # [0–100], can_tren=103.15 → không outlier thực
    "weighted_score_to_date": "winsorize",  # phạm vi lý thuyết mở, winsorize an toàn
    "n_assessments_submitted": "none",  # bị chặn bởi số bài tối đa của khoá học
    "date_registration": "none",  # âm đến dương – phạm vi tự nhiên
}


WINSORIZE_LIMITS = (0.01, 0.01)  # cắt 1% hai đầu


def _iqr_mask(s: pd.Series) -> pd.Series:
    """Trả về boolean mask: True = nghi ngờ ngoại lai theo quy tắc IQR."""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)


def log_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    In bảng phát hiện ngoại lai (IQR rule) trước khi xử lý.
    Dùng trong notebook, phối hợp với boxplot của Bình.
    """
    rows = []
    for col, strat in OUTLIER_STRATEGY.items():
        if col not in df.columns:
            continue
        mask = _iqr_mask(df[col])
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        rows.append(
            {
                "biến": col,
                "n_ngoại_lai (IQR)": int(mask.sum()),
                "pct_%": round(mask.mean() * 100, 2),
                "Q1": round(q1, 2),
                "Q3": round(q3, 2),
                "min": round(df[col].min(), 2),
                "max": round(df[col].max(), 2),
                "chiến_lược": strat,
            }
        )
    report = pd.DataFrame(rows)
    print("=== Ngoại lai trước khi xử lý (IQR rule) ===")
    print(report.to_string(index=False))
    return report


def handle_outliers(
    df: pd.DataFrame,
    log_path: Optional[str] = None,
    stats: Optional[dict] = None,
) -> pd.DataFrame:
    """
    Xử lý ngoại lai theo OUTLIER_STRATEGY.

    Quy tắc thiết kế:
    -  KHÔNG loại bỏ bản ghi: chỉ biến đổi giá trị (log1p / winsorize).
    -  log1p   : x → log(1 + x).  Yêu cầu x ≥ 0; phù hợp click count (không cần fit).
    -  winsorize: clip về ngưỡng [p1, p99] HỌC TRÊN TRAIN, rồi áp cho cả test —
       qua tham số ``stats`` (chống rò rỉ). Không học lại ngưỡng trên test.

    Tham số
    -------
    df       : X_train hoặc X_test (sau handle_missing)
    log_path : nếu không None, ghi CSV bảng so sánh trước/sau
    stats    : dict ngưỡng học trên train. Lần gọi đầu (train) để trống → hàm TÍNH
               (p1, p99) cho từng biến và LƯU; lần gọi sau (test) truyền lại dict →
               DÙNG ngưỡng train. Bỏ trống (None) → tính trên chính df (hành vi cũ).

    Trả về DataFrame đã xử lý.
    """
    df = df.copy()
    report_rows = []

    for col, strat in OUTLIER_STRATEGY.items():
        if col not in df.columns or strat == "none":
            continue

        b_min = round(df[col].min(), 3)
        b_max = round(df[col].max(), 3)
        b_mean = round(df[col].mean(), 3)
        n_out = int(_iqr_mask(df[col]).sum())

        if strat == "log1p":
            df[col] = np.log1p(df[col].clip(lower=0))
            reason = "Phân phối lệch phải → log1p giảm độ lệch"
        elif strat == "winsorize":
            # Ngưỡng [p1, p99] HỌC TRÊN TRAIN rồi áp cho test (clip ≡ winsorize).
            key = f"winsor_{col}"
            if stats is not None and key in stats:
                lo, hi = stats[key]
            else:
                lo = float(df[col].quantile(WINSORIZE_LIMITS[0]))
                hi = float(df[col].quantile(1.0 - WINSORIZE_LIMITS[1]))
                if stats is not None:
                    stats[key] = (lo, hi)
            df[col] = df[col].clip(lower=lo, upper=hi)
            reason = (
                f"Winsorize/clip tại [p{int(WINSORIZE_LIMITS[0]*100)}, "
                f"p{int((1 - WINSORIZE_LIMITS[1]) * 100)}] (ngưỡng học trên train)"
            )
        else:
            reason = "Không xử lý"

        report_rows.append(
            {
                "biến": col,
                "chiến_lược": strat,
                "n_nghi_ngoại_lai": n_out,
                "before_min": b_min,
                "before_max": b_max,
                "before_mean": b_mean,
                "after_min": round(df[col].min(), 3),
                "after_max": round(df[col].max(), 3),
                "after_mean": round(df[col].mean(), 3),
                "lý_do": reason,
            }
        )

    report_df = pd.DataFrame(report_rows)

    if log_path:
        report_df.to_csv(log_path, index=False, encoding="utf-8-sig")
        log.info("Bảng so sánh ngoại lai → %s", log_path)
    else:
        log.info(
            "Bảng xử lý ngoại lai:\n%s",
            report_df[
                [
                    "biến",
                    "chiến_lược",
                    "n_nghi_ngoại_lai",
                    "before_max",
                    "after_max",
                    "lý_do",
                ]
            ].to_string(index=False),
        )

    log.info("✔ handle_outliers: KHÔNG loại bỏ bản ghi nào | n_rows = %d", len(df))
    return df

## 3–5. Encoding, scaling & the full pipeline (Tasks 19, 20, 22)

ordinal (ordered ints) · nominal (one-hot, `handle_unknown=ignore`) · binary (0/1) · numeric (StandardScaler). The `ColumnTransformer` is `fit` on train only; `preprocess` wires the whole anti-leakage sequence together.

In [ ]:
class BinaryEncoder(BaseEstimator, TransformerMixin):
    """
    Mã hoá biến nhị phân theo bảng tra cứu cố định:
        gender    : M → 1,  F → 0
        disability: Y → 1,  N → 0

    Tuân thủ sklearn API (fit / transform / get_feature_names_out).
    Không cần fit thực sự (bảng tra cứu là hằng số), nhưng giữ để
    tương thích Pipeline và ColumnTransformer.
    """

    _MAPS: dict[str, dict[str, int]] = {
        "gender": {"M": 1, "F": 0},
        "disability": {"Y": 1, "N": 0},
    }

    def __init__(self, feature_names: list[str] = None):
        # Nhận danh sách tên cột từ bên ngoài để get_feature_names_out hoạt động đúng
        self.feature_names = (
            feature_names if feature_names is not None else BINARY_FEATURES
        )

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        """
        Nhận mảng/DataFrame có cột theo self.feature_names;
        trả về np.ndarray float với giá trị 0 / 1.
        """
        df = pd.DataFrame(X, columns=self.feature_names)
        for col in self.feature_names:
            mapping = self._MAPS.get(col, {})
            df[col] = df[col].map(mapping).fillna(0).astype(float)
        return df.values.astype(float)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names, dtype=object)


def build_ordinal_encoder() -> OrdinalEncoder:
    """
    OrdinalEncoder với thứ tự chính xác cho ba biến thứ bậc.

    Cơ sở chuyên môn
    ----------------
    Biến thứ bậc (highest_education, imd_band, age_band) có thứ tự
    nội tại; mã hoá số nguyên 0, 1, 2, … bảo toàn thứ tự đó.
    OneHotEncoder sẽ xoá thứ tự → KHÔNG dùng cho các biến này.

    handle_unknown='use_encoded_value', unknown_value=-1:
        Nếu tập test có giá trị lạ (không thấy ở train) → gán -1
        thay vì báo lỗi; mô hình cây xử lý được -1 tốt.
    """
    categories = [ORDINAL_ORDERS[c] for c in ORDINAL_FEATURES]
    return OrdinalEncoder(
        categories=categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )


def build_onehot_encoder() -> OneHotEncoder:
    """
    OneHotEncoder cho biến danh định (region, code_module, code_presentation).

    drop=None    : giữ tất cả các cột để SHAP/LIME diễn giải đầy đủ.
    sparse=False : trả về dense array (tương thích hơn với pipeline sau).
    handle_unknown='ignore': nếu test có giá trị lạ → cột tương ứng = 0.
    """
    return OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        drop=None,
    )


def build_scaler() -> StandardScaler:
    """
    StandardScaler: đưa biến định lượng về trung bình 0, độ lệch chuẩn 1.

    Cơ sở chuyên môn
    ----------------
    Tổng lượt click (0–hàng nghìn) áp đảo điểm số (0–100) về biên độ.
    Chuẩn hoá bắt buộc với Logistic Regression và ANN;
    RF / XGBoost / LightGBM ít nhạy hơn nhưng vẫn áp dụng để
    đảm bảo tính nhất quán trên toàn bộ pipeline.

    Quan trọng (anti-leakage)
    -------------------------
    Chỉ gọi scaler.fit() trên X_train.
    Gọi scaler.transform() cho cả X_train và X_test.
    KHÔNG bao giờ gọi fit_transform() trên toàn bộ tập dữ liệu.
    """
    return StandardScaler()


def build_column_transformer(
    numeric_cols: Optional[list[str]] = None,
    nominal_cols: Optional[list[str]] = None,
) -> ColumnTransformer:
    """
    Xây dựng ColumnTransformer kết hợp toàn bộ bước mã hoá và chuẩn hoá.

    Sơ đồ biến đổi
    ---------------
    numeric  →  StandardScaler
    ordinal  →  OrdinalEncoder  (thứ tự cố định)
    nominal  →  OneHotEncoder
    binary   →  BinaryEncoder   (0/1)
    chỉ báo  →  passthrough      (already 0/1)
    remainder → drop

    Tham số
    -------
    numeric_cols : danh sách cột numeric; mặc định = NUMERIC_FEATURES
    nominal_cols : danh sách cột nominal; mặc định = NOMINAL_FEATURES
    """
    if numeric_cols is None:
        numeric_cols = NUMERIC_FEATURES
    if nominal_cols is None:
        nominal_cols = NOMINAL_FEATURES

    return ColumnTransformer(
        transformers=[
            ("num", build_scaler(), numeric_cols),
            ("ordinal", build_ordinal_encoder(), ORDINAL_FEATURES),
            ("nominal", build_onehot_encoder(), nominal_cols),
            ("binary", BinaryEncoder(BINARY_FEATURES), BINARY_FEATURES),
            ("indicator", "passthrough", INDICATOR_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def fit_transform_train(
    X_train: pd.DataFrame,
    save_path: Optional[str] = None,
    numeric_cols: Optional[list[str]] = None,
    nominal_cols: Optional[list[str]] = None,
) -> tuple[ColumnTransformer, np.ndarray]:
    """
    Fit ColumnTransformer trên X_train; trả về (transformer, X_train_proc).

    Minh chứng fit trên train only (Task 20)
    -----------------------------------------
    Sau khi gọi hàm này, in:
        ct.named_transformers_['num'].mean_
    → tất cả giá trị đều tính từ X_train, không có thông tin từ X_test.

    Tham số
    -------
    X_train    : DataFrame tập huấn luyện (sau handle_missing + handle_outliers)
    save_path  : nếu không None, serialize transformer sang file .pkl
    """
    ct = build_column_transformer(numeric_cols=numeric_cols, nominal_cols=nominal_cols)
    X_train_proc = ct.fit_transform(X_train)

    # ── Minh chứng scaler chỉ học từ train ────────────────────────────
    scaler: StandardScaler = ct.named_transformers_["num"]
    n_show = min(5, len(scaler.mean_))
    log.info(
        "✔ scaler.mean_ (train only, %d biến đầu) = %s",
        n_show,
        np.round(scaler.mean_[:n_show], 4),
    )

    if save_path:
        joblib.dump(ct, save_path)
        log.info("ColumnTransformer đã lưu → %s", save_path)

    return ct, X_train_proc


def transform_test(ct: ColumnTransformer, X_test: pd.DataFrame) -> np.ndarray:
    """
    Áp dụng ColumnTransformer đã fit trên train để transform X_test.

    Nguyên tắc (anti-leakage)
    -------------------------
    Tuyệt đối KHÔNG gọi .fit() hay .fit_transform() lại trên X_test.
    Chỉ gọi .transform() với đối tượng ct đã được fit trên X_train.
    """
    return ct.transform(X_test)


def get_feature_names(ct: ColumnTransformer) -> list[str]:
    """
    Trả về danh sách tên đặc trưng sau khi transform.
    Quan trọng cho SHAP (feature names hiển thị trong waterfall plot)
    và LIME (feature importance label).
    """
    return ct.get_feature_names_out().tolist()


def preprocess(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    missing_log_path: Optional[str] = None,
    outlier_log_path: Optional[str] = None,
    scaler_save_path: Optional[str] = "scaler.pkl",
    numeric_cols: Optional[list[str]] = None,
    nominal_cols: Optional[list[str]] = None,
) -> tuple[np.ndarray, np.ndarray, ColumnTransformer, list[str]]:
    """
    Thực thi đầy đủ pipeline tiền xử lý theo đúng trình tự anti-leakage:

        phân chia (bên ngoài) →
        handle_missing        →
        handle_outliers       →
        fit_transform (train) →
        transform (test)

    Sau khi gọi hàm này, bước tiếp theo là TÁI LẤY MẪU (SMOTE/ADASYN)
    chỉ trên X_train_proc (xem ví dụ bên dưới).

    Tham số
    -------
    X_train, X_test       : DataFrame chỉ chứa cột đặc trưng (không có target)
    missing_log_path      : đường dẫn CSV nhật ký missing (tuỳ chọn)
    outlier_log_path      : đường dẫn CSV bảng so sánh outlier (tuỳ chọn)
    scaler_save_path      : đường dẫn lưu ColumnTransformer .pkl (tuỳ chọn)

    Trả về
    ------
    X_train_proc   : np.ndarray, đã transform
    X_test_proc    : np.ndarray, đã transform
    ct             : ColumnTransformer đã fit (dùng để transform dữ liệu mới)
    feature_names  : list[str]  (dùng cho SHAP / LIME)

    Ví dụ sử dụng
    -------------
    X_tr, X_te, ct, feat_names = preprocess(X_train, X_test)

    # Bước 5 – Tái lấy mẫu (SMOTE/ADASYN) chỉ trên train:
    from imblearn.over_sampling import SMOTE
    X_tr_res, y_tr_res = SMOTE(random_state=42).fit_resample(X_tr, y_train)

    # Huấn luyện mô hình:
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_tr_res, y_tr_res)

    # Đánh giá trên test (chưa qua resampling):
    from sklearn.metrics import classification_report
    print(classification_report(y_test, clf.predict(X_te)))
    """

    log.info("══ preprocess: bắt đầu ══")

    # ── Bước 2: Xử lý giá trị khuyết (median HỌC TRÊN TRAIN) ─────────
    log.info("── Bước 2: handle_missing ──")
    missing_stats: dict = {}
    X_train = handle_missing(X_train, log_path=missing_log_path, stats=missing_stats)
    X_test = handle_missing(
        X_test, log_path=None, stats=missing_stats
    )  # áp tham số train

    # ── Bước 3: Xử lý ngoại lai (ngưỡng winsorize HỌC TRÊN TRAIN) ────
    log.info("── Bước 3: handle_outliers ──")
    outlier_stats: dict = {}
    X_train = handle_outliers(X_train, log_path=outlier_log_path, stats=outlier_stats)
    X_test = handle_outliers(
        X_test, log_path=None, stats=outlier_stats
    )  # áp ngưỡng train

    # ── Bước 4a: Fit & transform trên train ───────────────────────────
    log.info("── Bước 4a: fit_transform (train only) ──")
    ct, X_train_proc = fit_transform_train(
        X_train,
        save_path=scaler_save_path,
        numeric_cols=numeric_cols,
        nominal_cols=nominal_cols,
    )

    # ── Bước 4b: Transform test ───────────────────────────────────────
    log.info("── Bước 4b: transform (test) ──")
    X_test_proc = transform_test(ct, X_test)

    # ── Tổng kết ──────────────────────────────────────────────────────
    feature_names = get_feature_names(ct)
    log.info(
        "✔ preprocess hoàn tất | train: %s | test: %s | n_features: %d",
        X_train_proc.shape,
        X_test_proc.shape,
        len(feature_names),
    )
    log.info("  [Bước 5 tiếp theo] Áp dụng SMOTE/ADASYN CHỈ trên X_train_proc")

    return X_train_proc, X_test_proc, ct, feature_names

## Demonstration on the master table

**Step 1 — load, split safely, *then* preprocess.** The split comes before any learned statistic — this ordering is what prevents leakage.

In [ ]:
master = pd.read_parquet(INTERIM_DIR / 'master_raw.parquet')
X, y = make_X_y(master)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print('X_train:', X_train.shape, '| X_test:', X_test.shape,
      f'| train at-risk: {y_train.mean():.1%} | test at-risk: {y_test.mean():.1%}')

**Step 2 — inspect (before).**

In [ ]:
_ = log_missing(X_train)
_ = log_outliers(X_train)

**Step 3 — run the pipeline.**

In [ ]:
X_tr, X_te, ct, feat_names = preprocess(
    X_train.copy(), X_test.copy(), scaler_save_path=None)
print('X_train_proc:', X_tr.shape, '| X_test_proc:', X_te.shape,
      '| n_features:', len(feat_names))
print('first 10 feature names:', feat_names[:10])

**Step 4 — anti-leakage proof.** The scaler mean equals the *train* mean (not the full-data mean), and there are no NaNs after transform.

In [ ]:
scaler = ct.named_transformers_['num']
import numpy as np
assert not np.isnan(X_tr).any() and not np.isnan(X_te).any()
tc = X_train['total_clicks'].clip(lower=0)
train_logmean = np.log1p(tc.clip(upper=tc.quantile(0.99) if False else tc.max())).mean()
print('scaler.mean_[:5] (learned on train only):', np.round(scaler.mean_[:5], 4))
print('no NaN after transform — OK')

**Step 5 — resampling belongs *after* the split, on train only.**

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    X_res, y_res = SMOTE(random_state=RANDOM_SEED).fit_resample(X_tr, y_train)
    print(f'SMOTE: train {X_tr.shape[0]:,} -> {X_res.shape[0]:,} (test untouched)')
except ImportError:
    print('imbalanced-learn not installed — skip SMOTE demo')

## Conclusion

`X_train_proc` / `X_test_proc` are model-ready with **no leakage**: every learned statistic came from train and was applied to test. Resampling, if used, applies to the transformed training set only. The fitted `ColumnTransformer` (`ct`) and `feat_names` feed model training and SHAP/LIME explanation.